In [1]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [2]:
ground_truth[0]

{'question': 'I just found this course late — can I still sign up and follow along, or is it too late?',
 'document': '74eb249bbf'}

In [3]:
# faq documents and search index
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

[{'course': 'machine-learning-zoomcamp', 'course_name': 'ML Zoomcamp', 'path': '/json/machine-learning-zoomcamp.json', 'questions_count': 471}, {'course': 'mlops-zoomcamp', 'course_name': 'MLOps Zoomcamp', 'path': '/json/mlops-zoomcamp.json', 'questions_count': 253}, {'course': 'stock-markets-analytics-zoomcamp', 'course_name': 'Stock Markets Analytics Zoomcamp', 'path': '/json/stock-markets-analytics-zoomcamp.json', 'questions_count': 93}, {'course': 'ai-dev-tools-zoomcamp', 'course_name': 'AI Dev Tools Zoomcamp', 'path': '/json/ai-dev-tools-zoomcamp.json', 'questions_count': 41}, {'course': 'data-engineering-zoomcamp', 'course_name': 'Data Engineering Zoomcamp', 'path': '/json/data-engineering-zoomcamp.json', 'questions_count': 404}, {'course': 'llm-zoomcamp', 'course_name': 'LLM Zoomcamp', 'path': '/json/llm-zoomcamp.json', 'questions_count': 118}]


In [4]:
# create the lookup table
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

In [5]:
# toyaikit - handles the agent loop and stores the full message history.
from dotenv import load_dotenv
from openai import OpenAI
from toyaikit.llm import OpenAIClient

load_dotenv()
openai_client = OpenAI()

In [6]:
# define the search tool
def search_tool(query: str) -> list[dict]:
    """Search the FAQ documents for the query."""
    return index.search(
      query,
      num_results=5,
      boost_dict={"question": 1.0, "answer": 2.0, "section": 0.1},
      filter_dict={"course": "llm-zoomcamp"}
    )

In [7]:
# create the runner
from toyaikit.tools import Tools
from toyaikit.chat.runners import OpenAIResponsesRunner

agent_tools = Tools()
agent_tools.add_tool(search_tool)

instructions = """
You're a course teaching assistant. Answer student questions based on
the FAQ search results. Use the search tool before answering.
""".strip()

runner = OpenAIResponsesRunner(
  developer_prompt=instructions,
  llm_client=OpenAIClient(model="gpt-5.4-mini"),
  tools=agent_tools,
)

In [8]:
rec = ground_truth[0]

result = runner.loop(prompt=rec["question"])

In [9]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant. Answer student questions based on\nthe FAQ search results. Use the search tool before answering.", role='developer', phase=None, type=None),
 EasyInputMessage(content='I just found this course late — can I still sign up and follow along, or is it too late?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"late enrollment sign up follow along too late course FAQ"}', call_id='call_vYJa2Vomtnjg9ZfzN0gWlaOI', name='search_tool', type='function_call', id='fc_0a393c9b8c2f8650006a66831fc6d4819586c35915a33f5ab3', caller=None, namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_vYJa2Vomtnjg9ZfzN0gWlaOI',
  'output': '[\n  {\n    "id": "04919992b3",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "How should I start the course and follow the weekly workflow?",\n    "answer": "Start with the [LLM Zoomcamp docs]

In [12]:
# For this lesson, the trajectory is only the tool calls. We don't need to send the full message history to the judge.

def extract_tool_calls(messages):
  tool_calls = []

  for message in messages:
    if isinstance(message, dict):
      continue

    if message.type == "function_call":
      tool_calls.append(
        {
          "name": message.name,
          "arguments": message.arguments,
        }
      )

  return tool_calls

In [13]:
tool_calls = extract_tool_calls(result.all_messages)
tool_calls

[{'name': 'search_tool',
  'arguments': '{"query":"late enrollment sign up follow along too late course FAQ"}'}]

In [14]:
# get the original answer
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

In [16]:
import json
agent_result = {
  "question": rec["question"],
  "answer_agent": result.last_message,
  "answer_orig": answer_orig,
  "tool_calls": json.dumps(tool_calls),
  "cost": result.cost.total_cost,
  "document": doc_id,
}
agent_result


{'question': 'I just found this course late — can I still sign up and follow along, or is it too late?',
 'answer_agent': 'Yes — you can still sign up and follow along.\n\nThe course materials and videos are available, and the FAQ says you can start whenever you want. You can work through the lessons at your own pace even if you joined late.\n\nA couple of caveats:\n- Homework has deadlines, and there are no late submissions once the form closes.\n- If you want a certificate, you need to finish with a live cohort, since certificates aren’t available in self-paced mode.\n\nIf you want, I can also point you to the best place to start in the course.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'tool_calls': '[{"name": "search_tool", "arguments": "{\\"query\\":\\"late enrollment sign up follow along too late course FAQ\\"}"}]',
 'cost': Decimal('0.00143175'),
 'document': '74eb249bbf'}

In [17]:
def generate_agent_answer(rec):
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    result = runner.loop(prompt=rec["question"])

    tool_calls = extract_tool_calls(result.all_messages)

    answer_record = {
        "question": rec["question"],
        "answer_agent": result.last_message,
        "answer_orig": original_doc["answer"],
        "tool_calls": json.dumps(tool_calls),
        "cost": result.cost.total_cost,
        "document": doc_id,
    }

    return answer_record

In [18]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

with ThreadPoolExecutor(max_workers=6) as pool:
    agent_answers = map_progress(pool, ground_truth[:50], generate_agent_answer)


  0%|          | 0/50 [00:00<?, ?it/s]

In [19]:
df_agent = pd.DataFrame(agent_answers)
df_agent.head()

,question,answer_agent,answer_orig,tool_calls,cost,document
0,I just found this course late — can I still si...,Yes — you can still join and follow along even...,"Yes, but if you want to receive a certificate,...","[{""name"": ""search_tool"", ""arguments"": ""{\""quer...",0.0012735,74eb249bbf
1,"If I join after the course already started, ca...",Yes — you can still join after the course has ...,"Yes, but if you want to receive a certificate,...","[{""name"": ""search_tool"", ""arguments"": ""{\""quer...",0.00127875,74eb249bbf
2,Do I have to submit the final project before s...,"Yes — to be eligible for the certificate, you ...","Yes, but if you want to receive a certificate,...","[{""name"": ""search_tool"", ""arguments"": ""{\""quer...",0.0010305,74eb249bbf
3,Is it okay to start the course now if I missed...,Yes — you can start the course now even if you...,"Yes, but if you want to receive a certificate,...","[{""name"": ""search_tool"", ""arguments"": ""{\""quer...",0.00270525,74eb249bbf
4,What’s the deadline for project submission if ...,"To get the course certificate, you need to sub...","Yes, but if you want to receive a certificate,...","[{""name"": ""search_tool"", ""arguments"": ""{\""quer...",0.0009945,74eb249bbf


In [20]:
df_agent["cost"].sum()

Decimal('0.06800550')

In [21]:
df_agent.to_csv("data/agent-answers.csv", index=False)

In [22]:
df_agent = pd.read_csv("data/agent-answers.csv")
agent_answers = df_agent.to_dict(orient="records")

In [23]:
# judge output type with two scores
from pydantic import BaseModel, Field
from typing import Literal

class AgentEvaluation(BaseModel):
    answer_reasoning: str = Field(
        description="Reasoning about whether the final answer is correct."
    )
    answer_score: Literal["good", "bad"] = Field(
        description="'good' if the final answer matches the original answer."
    )
    trajectory_reasoning: str = Field(
        description="Reasoning about whether the tool calls were useful."
    )
    trajectory_score: Literal["good", "bad"] = Field(
        description="'good' if the tool calls were reasonable for the question."
    )

In [24]:
agent_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI agent
4. The tool calls made by the agent

Evaluate two things:

Answer quality:
- Does the agent answer match the original answer?
- It does not need to be word-for-word identical.
- It should contain the same key information.

Trajectory quality:
- Were the search queries relevant to the question?
- Did the queries include important keywords from the question?
- Did the agent avoid duplicate or unnecessary tool calls?
- If it made multiple searches, did the later searches refine the query?
- Was the number of search calls reasonable? Usually 1 is enough, 2-3
  can be okay, and more than 3 needs a clear reason.
- Did the tool calls support the final answer?

Mark answer_score as 'good' if the final answer is correct.
Mark trajectory_score as 'good' if the tool calls were reasonable.
""".strip()

agent_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

Agent Answer:
{answer_agent}

Tool Calls:
{tool_calls}
""".strip()

In [27]:
import json
from evaluation_utils import calc_total_price, llm_structured_retry

def evaluate_agent_answer(rec, model="gpt-5.4-mini"):
    tool_calls = rec["tool_calls"]

    if isinstance(tool_calls, str):
        tool_calls = json.loads(tool_calls)

    prompt = agent_judge_prompt.format(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_agent=rec["answer_agent"],
        tool_calls=json.dumps(tool_calls, indent=2),
    )

    result, usage = llm_structured_retry(
        openai_client,
        agent_judge_instructions,
        prompt,
        AgentEvaluation,
        model=model,
    )

    return result, usage

In [28]:
agent_eval, usage = evaluate_agent_answer(agent_answers[0])

agent_eval

AgentEvaluation(answer_reasoning='The agent’s answer matches the ground truth. It correctly says the student can still join and follow along, and it includes the key condition for earning a certificate: the project must be submitted while submissions are still being accepted. This is consistent with the original answer, even if phrased a bit more expansively.', answer_score='good', trajectory_reasoning='The single search query is relevant to the question and contains the core idea of being late to the course. One search call is reasonable, and there were no unnecessary duplicate calls. The tool use supports the final answer.', trajectory_score='good')

In [29]:
def judge_agent_record(rec):
    agent_eval, usage = evaluate_agent_answer(rec)

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "answer_score": agent_eval.answer_score,
        "answer_reasoning": agent_eval.answer_reasoning,
        "trajectory_score": agent_eval.trajectory_score,
        "trajectory_reasoning": agent_eval.trajectory_reasoning,
    }

    return result, usage

In [31]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, agent_answers, judge_agent_record)

  0%|          | 0/50 [00:00<?, ?it/s]

In [32]:
agent_evaluations = []
usages = []

for evaluation, usage in results:
    agent_evaluations.append(evaluation)
    usages.append(usage)

In [33]:
df_agent_eval = pd.DataFrame(agent_evaluations)

In [34]:
calc_total_price(usages)

0.0554625

In [35]:
df_agent_eval["answer_score"].value_counts()

answer_score
good    45
bad      5
Name: count, dtype: int64

In [36]:
df_agent_eval["trajectory_score"].value_counts()

trajectory_score
good    50
Name: count, dtype: int64

In [37]:
df_agent_eval.to_csv("data/agent-evaluations.csv", index=False)